<div align="center">

# 📚 Müfredat Senkronizasyonu (Google Sheets → Web)

Bu notebook ile Google Sheets'te düzenlediğiniz **Konular, Ödevler, Kaynaklar**
bilgilerini tek tıkla web sayfasına yayınlarsınız.

**Akış:**
1. Google Sheets'te haftaların konularını/ödevlerini/kaynaklarını düzenlersiniz
2. Bu notebook Sheets'ten veriyi çeker → `curriculum.ts`'yi günceller
3. GitHub'a push eder → web sayfası 1-2 dakika içinde güncellenir

---
*Dr. Murat Altun · ECS Veri Bilimi ve YZ Uzmanlığı Programı*

</div>

## 1. Ayarlar

### Google Sheets Paylaşım ID'sini Al
1. Sheets'i aç → Paylaş → **Bağlantıya sahip olan herkes (Görüntüleyici)**
2. URL'den ID'yi kopyala: `docs.google.com/spreadsheets/d/`**`BURASI`**`/edit`
3. Aşağıdaki `SHEET_ID` değişkenine yapıştır

In [ ]:
# ===== AYARLAR =====
SHEET_ID = "1n7_pvEUX72hdzkO6JgIYnpZxOavkPEWJA8p-CjDU4G0"  # Google Sheets ID
GITHUB_REPO = "DrMuratAltun/VB-YZ-90"
BRANCH = "main"

## 2. Google Sheets'ten CSV İndir

In [ ]:
import urllib.request, os

CSV_URL = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv"
CSV_PATH = "/content/curriculum_sheet.csv"

urllib.request.urlretrieve(CSV_URL, CSV_PATH)
size = os.path.getsize(CSV_PATH)
print(f"✅ CSV indirildi: {size} bayt")

# İlk 3 satırı göster (kontrol için)
with open(CSV_PATH, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 3: break
        print(line[:200])

## 3. GitHub Repo'yu Klonla ve curriculum.ts'yi Güncelle

In [ ]:
# GitHub token (Colab Secrets'ta GITHUB_TOKEN olarak saklanmalı)
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    print("✅ Token Colab Secrets'tan alındı")
except Exception:
    GITHUB_TOKEN = input("GitHub Personal Access Token: ")

In [ ]:
import os, shutil, subprocess

WORK = "/content/_curriculum_sync"
if os.path.exists(WORK):
    shutil.rmtree(WORK)

clone_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, clone_url, WORK], check=True)

subprocess.run(["git", "-C", WORK, "config", "user.name", "Dr. Murat Altun"], check=True)
subprocess.run(["git", "-C", WORK, "config", "user.email", "emurataltun@gmail.com"], check=True)

print("✅ Repo klonlandı")

In [ ]:
# CSV'yi curriculum.ts'ye uygula
script = f"{WORK}/scripts/update_curriculum_from_csv.py"
result = subprocess.run(
    ["python3", script, CSV_PATH],
    capture_output=True, text=True, cwd=WORK
)
print(result.stdout)
if result.stderr:
    print("⚠️", result.stderr)

## 4. Değişiklikleri GitHub'a Gönder

In [ ]:
# Değişiklik var mı?
diff = subprocess.run(
    ["git", "-C", WORK, "diff", "--stat"],
    capture_output=True, text=True
)

if not diff.stdout.strip():
    print("ℹ️ Değişiklik yok — Sheets ve curriculum.ts senkronize.")
else:
    print("📊 Değişiklikler:")
    print(diff.stdout)
    
    subprocess.run(["git", "-C", WORK, "add", "web/src/data/curriculum.ts"], check=True)
    subprocess.run(
        ["git", "-C", WORK, "commit", "-m", "Müfredat güncellendi (Sheets → curriculum.ts)"],
        check=True
    )
    push = subprocess.run(
        ["git", "-C", WORK, "push"],
        capture_output=True, text=True
    )
    if push.returncode == 0:
        print("\n✅ GitHub'a gönderildi!")
        print("🌐 Web sayfası 1-2 dakika içinde güncellenecek:")
        print("   https://drmurataltun.github.io/VB-YZ-90/")
    else:
        print("❌ Push hatası:", push.stderr)

# Temizlik
shutil.rmtree(WORK)
print("\n🧹 Geçici dosyalar temizlendi.")

---

## 📋 Sheets Kolon Formatı

| Kolon | İçerik |
|-------|--------|
| `hafta_id` | 1-15 arası sayı |
| `baslik` | Sadece referans için (değiştirilmez) |
| `konular` | Her satıra bir konu (Alt+Enter ile yeni satır) |
| `odevler` | Her satıra bir ödev |
| `kaynaklar` | `Başlık | URL` formatında, her satıra bir kaynak |

**Örnek kaynaklar hücresi:**
```
Python Resmi Tutorial | https://docs.python.org/3/tutorial/
Google Colab | https://colab.research.google.com/
```

---

<div align="center">

**Dr. Murat Altun** · Veri Bilimi ve Yapay Zeka Eğitmeni

</div>